In [ ]:
import requests
import pandas as pd
from io import StringIO

# LINKS FOR DATA
BASE_URL  = "https://www.smard.de/app/chart_data"
NIP_URL   = "https://www.smard.de/nip-download-manager/nip/download/market-data"
REGION    = "DE"
RES       = "hour"

START = "2022-01-01"
END   = "2026-01-01"

# Generation per technology available in germany
SOURCES = {
    "BIO": 4066,
    "WON": 4067,
    "PV":  4068,
    "HC":  4069,
    "HPS": 4070,
    "FG":  4071,
    "WOF": 1225,
    "HP":  1226,
    "OR":  1228,
    "LIG": 1223,
    "OC":  1227,
}
# Energy Consumption
CONSUMPTION_SOURCES = {
    "TGL":  410,
    "GL":   410,
    "HPS":  4387,
    "RL":   4359,
}

# Wholesale market prices
WHOLESALE_MODULE_IDS = [
    8004169,   # Deutschland/Luxemburg [€/MWh]
]


# Function to get timestamps for fata request
def get_timestamps(filter_id):
    url = f"{BASE_URL}/{filter_id}/{REGION}/index_{RES}.json"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return r.json()["timestamps"]

#Filtering from webpage

def get_series(filter_id, timestamp):
    url = f"{BASE_URL}/{filter_id}/{REGION}/{filter_id}_{REGION}_{RES}_{timestamp}.json"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return r.json()["series"]


def fetch_source(name, filter_id, start_date, end_date):
    start_ts = int(pd.Timestamp(start_date, tz="Europe/Berlin").timestamp() * 1000)
    end_ts   = int(pd.Timestamp(end_date,   tz="Europe/Berlin").timestamp() * 1000)
    timestamps = get_timestamps(filter_id)
    relevant   = [t for t in timestamps if t <= end_ts]
    all_rows = []
    for ts in relevant:
        series = get_series(filter_id, ts)
        for row_ts, value in series:
            if start_ts <= row_ts <= end_ts:
                all_rows.append({
                    "timestamp": pd.Timestamp(row_ts, unit="ms", tz="Europe/Berlin"),
                    name: value
                })
    return pd.DataFrame(all_rows).set_index("timestamp")


# Function for wholesale market extraction

def fetch_wholesale_table(module_ids, start_date, end_date):
    start_ts = int(pd.Timestamp(start_date, tz="Europe/Berlin").timestamp() * 1000)
    end_ts   = int(pd.Timestamp(end_date,   tz="Europe/Berlin").timestamp() * 1000)

    payload = {
        "request_form": [{
            "format":          "CSV",
            "moduleIds":       module_ids,
            "region":          REGION,
            "timestamp_from":  start_ts,
            "timestamp_to":    end_ts,
            "type":            "discrete",
            "language":        "de"
        }]
    }

    print(f"  POST → {NIP_URL}")
    r = requests.post(NIP_URL, json=payload, timeout=120)
    r.raise_for_status()

    text = r.text
    if not text.strip() or text.strip().startswith("<"):
        raise ValueError(f"Respuesta inesperada: {text[:300]}")

    # CSV con ";" como separador y "," como decimal
    df = pd.read_csv(
        StringIO(text),
        sep=";",
        decimal=",",
        thousands=".",
        encoding="utf-8"
    )

    print(f"  Columnas recibidas: {list(df.columns)}")

    # Construir timestamp desde las columnas de fecha/hora
    if "Datum von" in df.columns and "Uhrzeit von" in df.columns:
        df["timestamp"] = pd.to_datetime(
            df["Datum von"].str.strip() + " " + df["Uhrzeit von"].str.strip(),
            dayfirst=True
        ).dt.tz_localize("Europe/Berlin", ambiguous="NaT", nonexistent="NaT")
    elif "Datum von" in df.columns:
        df["timestamp"] = pd.to_datetime(
            df["Datum von"].str.strip(), dayfirst=True
        ).dt.tz_localize("Europe/Berlin", ambiguous="NaT", nonexistent="NaT")
    else:
        # Usar primeras dos columnas como fecha y hora
        col0, col1 = df.columns[0], df.columns[1]
        df["timestamp"] = pd.to_datetime(
            df[col0].astype(str).str.strip() + " " + df[col1].astype(str).str.strip(),
            dayfirst=True
        ).dt.tz_localize("Europe/Berlin", ambiguous="NaT", nonexistent="NaT")

    df = df.set_index("timestamp")

    # Eliminar columnas de fecha/hora originales
    drop_cols = [c for c in df.columns if "Datum" in c or "Uhrzeit" in c]
    df = df.drop(columns=drop_cols, errors="ignore")

    # Convertir todo a numérico
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # El NIP devuelve cuartos de hora aunque se pida horario → filtrar :00
    df = df[df.index.minute == 0]


    return df


# FIRST: GENERATION PER TECHNOLOGY


print("BLOQUE 1: Generación por tecnología")


dfs = []
for name, fid in SOURCES.items():
    print(f"  Descargando {name}...")
    df = fetch_source(name, fid, START, END)
    dfs.append(df)

generation = pd.concat(dfs, axis=1)
generation.index.name = "timestamp"

# SECOND: ENERGY CONSUMPTION


print("BLOQUE 2: Energy consumption (TGL | GL | HPS | RL)")


already_fetched = {}
dfs_consumption = []

for name, fid in CONSUMPTION_SOURCES.items():
    if fid in already_fetched:
        df_cached = already_fetched[fid].copy()
        df_cached.columns = [name]
        dfs_consumption.append(df_cached)
        print(f"  ✓ {name} (filter_id={fid}) — reutilizado del caché")
    else:
        print(f"  Descargando {name} (filter_id={fid})...")
        try:
            df = fetch_source(name, fid, START, END)
            already_fetched[fid] = df
            dfs_consumption.append(df)

consumption = pd.concat(dfs_consumption, axis=1)
consumption.index.name = "timestamp"

# THIRD: WHOLESALE PRICES

print("BLOQUE 3: Wholesale prices [€/MWh]")

try:
    prices = fetch_wholesale_table(WHOLESALE_MODULE_IDS, START, END)
    prices.index.name = "timestamp"
    print(f"Precios: {prices.shape}")
    print(f"Columns: {list(prices.columns)}")
except Exception as e:
    print(f"Error descargando: {e}")
    prices = pd.DataFrame()

# COMPILING EVERYTHING

frames = [consumption, generation]
if not prices.empty:
    frames.append(prices)

result = pd.concat(frames, axis=1)
result.index.name = "timestamp"

print(result.head())
print(f"\nShape: {result.shape}")
print(f"Columnas ({len(result.columns)}): {list(result.columns)}")

# SAVING
result.to_csv("SMARTDE_2025.csv")
